In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = "https://finance.yahoo.com/markets/crypto/most-active/"
# there are different request methods: get (most popular and easy)
# stateful protocol - connection established and then closed
# stateless protocol - http, every request (like each other page opening) is separate.
# all data is sent in headers along with URL. Like session id to identify next request and source of accessing web page
# HTTP response status codes
# 100s -
# 200s - OK
# 300s - redirect
# 400s - client error (404 - not found, 429 - too many requests)

HEADERS = {'User-Agent': 'Mozilla/5.0'} #enough for the site to wok

def parse_human_readable(value):
    if not value:
        return None
    val = value.replace(',', '')
    last_char = val[-1]
    val_no_last_char = val[:-1]

    match (last_char):
        case 'K':
            return float(val_no_last_char)*1000
        case 'M':
            return float(val_no_last_char)*1_000_000
        case 'B':
            return float(val_no_last_char)*1_000_000_000
        case 'T':
            return float(val_no_last_char)*1_000_000_000_000
        case _:
            return float(val)

def crypto_volumes():
    response = requests.get(URL, headers=HEADERS) #question to get the info from URL
    print(response)
    print(response.status_code)
    html = response.text
    #print(html)
    soup = BeautifulSoup(html, 'html.parser')
    table = soup.find('table', attrs={'class': 'yf-19xszjc'}) #(this code changes once every while, so in some time it might no longer work)
    print(table)
    rows = table.find('tbody').find_all('tr')
    data = [] #to turn it into pandas dataframe
    for row in rows:
        print(row)
        print('-------------')
        cells = row.find_all('td')
        symbol = cells[0].text.strip()
        print(symbol)
        price = cells[4].text.strip()
        market_cap = cells[6].text.strip()
        volume = cells[8].text.strip()
        record = {
            'symbol': symbol,
            'price': parse_human_readable(price),
            'market_cap': market_cap,
            'volume': volume,
        }
        print(record)
        data.append(record)
    df = pd.DataFrame(data)
    df['market_cap'] = df['market_cap'].apply(parse_human_readable)
    df['volume'] = df['volume'].apply(parse_human_readable)
    print(df)

def save_crypto_volumes(df):
    df.to_csv('data/crypto_volumes.csv', index=False)
    df.to_json('data/crypto_volumes.json', orient='records')
    df.to_xlsx('data/crypto_volumes.xlsx', index=False)

In [ ]:
def x2(x):
    return x*2

print(x2(1000))
x_2 = lambda x:x*2 #lambda expression shows there is a function that turns x into x*2
print(x_2(1000))

In [ ]:
df = crypto_volumes()
save_crypto_volumes(df)
#protocol = set of rules to communicate
#this is manually hard to read, extracting data via BeautifulSoup